# 01 EDA: decisions for `features.py`

This notebook answers six questions. Each one ends in a decision to record in `docs/decisions.md`.

**Rules**
- Only train and validation rows are loaded (`load_dev_frame`). The test period stays unseen; drift on test is stage 3's job.
- Preprocessing choices are made from **train**. Validation is used only to check that a choice holds up later in time (for example, unseen categories).
- Exploration only. Anything that becomes part of the pipeline moves into `src/fraudpriv/`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir("..")

from fraudpriv.data import AMOUNT_COL, ID_COL, TARGET, TIME_COL, load_dev_frame
from fraudpriv.utils import load_config

cfg = load_config("configs/split.yaml")
df = load_dev_frame(cfg["data_dir"], cfg["split_dir"])
train = df[df["split"] == "train"]
val = df[df["split"] == "val"]

# Validated palette pair (blue / orange) on a near-white surface; legit vs fraud.
LEGIT, FRAUD, MUTED = "#2a78d6", "#eb6834", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.grid": True, "grid.color": "#ebeae6",
    "grid.linewidth": 0.6, "axes.titleweight": "bold", "lines.linewidth": 2,
    "text.color": "#0b0b0b", "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED, "figure.dpi": 110,
})
pd.set_option("display.max_rows", 60)

In [ ]:
FEATURES = [c for c in df.columns if c not in (ID_COL, TARGET, TIME_COL, "split")]
print(f"rows: train {len(train):,}, val {len(val):,}; candidate features: {len(FEATURES)}")
df.groupby("split")[TARGET].agg(n="size", frauds="sum", fraud_rate="mean")

## Q1. Missingness

How much is missing, does missingness come in blocks of columns, and is *being missing* predictive of fraud?

Decides: which columns to drop, the imputation rule, and whether to add missing-value indicators (one per column or one per block).

In [ ]:
miss = train[FEATURES].isna().mean().sort_values(ascending=False)
print(f"columns >90% missing: {(miss > 0.9).sum()}, >50%: {(miss > 0.5).sum()}, none missing: {(miss == 0).sum()}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(miss, bins=40, color=LEGIT, edgecolor="#fcfcfb", linewidth=1)
ax.set(title="Missing rate per column (train)", xlabel="share of rows missing", ylabel="columns")
plt.show()

In [ ]:
# Columns with an identical missingness pattern form blocks (common for V and id_ columns).
mask = train[FEATURES].isna()
pattern = pd.Series({c: hash(mask[c].to_numpy().tobytes()) for c in FEATURES})
blocks = (
    pd.DataFrame({"col": FEATURES, "pattern": pattern[FEATURES].to_numpy(), "miss": miss[FEATURES].to_numpy()})
    .query("miss > 0")
    .groupby("pattern")
    .agg(n_cols=("col", "size"), miss_rate=("miss", "first"), example=("col", lambda s: ", ".join(s[:4])))
    .sort_values("n_cols", ascending=False)
)
print(f"{(miss > 0).sum()} columns with missing values fall into {len(blocks)} distinct missingness patterns")
blocks.head(15)

In [ ]:
# Fraud rate when a column is missing vs present (train). Large gaps mean missingness carries signal.
rows = []
for c in miss[(miss > 0.01) & (miss < 0.99)].index:
    m = mask[c]
    rows.append((c, miss[c], train.loc[m, TARGET].mean(), train.loc[~m, TARGET].mean()))
gap = pd.DataFrame(rows, columns=["col", "miss_rate", "fraud_if_missing", "fraud_if_present"])
gap["diff"] = gap["fraud_if_missing"] - gap["fraud_if_present"]
gap.reindex(gap["diff"].abs().sort_values(ascending=False).index).head(20)

**Decision (Q1):** _drop threshold, imputation rule (e.g. median for numeric, fit on train), indicators per column or per block._

## Q2. Categorical columns

Cardinality on train, and how often validation contains values never seen in train.

Decides: encoding (one-hot with a frequency floor, frequency encoding, or target encoding) and how unseen values are handled. The encoding must suit the MLP and the DP model, not only LightGBM.

In [ ]:
CATEGORICAL = (
    ["ProductCD", "P_emaildomain", "R_emaildomain", "DeviceType", "DeviceInfo"]
    + [f"card{i}" for i in range(1, 7)] + ["addr1", "addr2"]
    + [f"M{i}" for i in range(1, 10)] + [f"id_{i}" for i in range(12, 39)]
)
CATEGORICAL = [c for c in CATEGORICAL if c in df.columns]

def unseen_share(c):
    seen = set(train[c].dropna().unique())
    v = val[c].dropna()
    return (~v.isin(seen)).mean() if len(v) else np.nan

cat = pd.DataFrame({
    "n_unique_train": [train[c].nunique() for c in CATEGORICAL],
    "miss_rate_train": [train[c].isna().mean() for c in CATEGORICAL],
    "val_rows_unseen": [unseen_share(c) for c in CATEGORICAL],
}, index=CATEGORICAL).sort_values("n_unique_train", ascending=False)
cat

In [ ]:
# How many one-hot columns survive a minimum-frequency floor? (Feeds Q6.)
for floor in (0.0001, 0.001, 0.01):
    n = sum((train[c].value_counts(normalize=True) >= floor).sum() + 1 for c in CATEGORICAL)  # +1 = other/unseen
    print(f"one-hot width with frequency floor {floor:.2%}: {n:,}")

**Decision (Q2):** _encoding per column group, frequency floor, handling of unseen values._

## Q3. Leakage and time-dependent features

- `TransactionDT` is a time offset; it is excluded as a feature.
- `D` columns are day counts relative to the transaction, so their raw values drift with time.
- A rough card ID can be rebuilt as `card1` + `addr1` + (transaction day - `D1`). If many validation rows belong to cards seen in train, a model can memorize cards, which flatters results.

Decides: which `D` columns are used and in what form, and whether card-level aggregates or a card ID are allowed.

In [ ]:
day = df[TIME_COL] // 86400
D_COLS = [c for c in df.columns if c.startswith("D") and c[1:].isdigit()]

# Correlation of each D column with time on train: high values mean the raw column drifts with the clock.
drift = pd.Series({c: train[c].corr(train[TIME_COL]) for c in D_COLS}).sort_values(key=abs, ascending=False)
drift.rename("corr_with_time").to_frame()

In [ ]:
uid = df["card1"].astype(str) + "_" + df["addr1"].astype(str) + "_" + (day - df["D1"]).astype(str)
is_train = (df["split"] == "train").to_numpy()
uid_train = set(uid[is_train])
in_train = uid[~is_train].isin(uid_train).to_numpy()
val_fraud = val[TARGET].to_numpy() == 1
print(f"val rows whose rebuilt card ID also appears in train: {in_train.mean():.1%}")
print(f"val frauds whose rebuilt card ID also appears in train: {in_train[val_fraud].mean():.1%}")

# Label consistency within a card on train: if fraud cards are almost always all-fraud,
# memorizing the card is a shortcut that the time split does not remove.
g = train.assign(uid=uid[is_train].to_numpy()).groupby("uid")[TARGET].agg(["size", "mean"])
multi = g[g["size"] > 1]
print(f"cards with >1 train transaction: {len(multi):,}; of those with any fraud, all-fraud share: "
      f"{(multi.loc[multi['mean'] > 0, 'mean'] == 1).mean():.1%}")

**Decision (Q3):** _which D columns (raw, normalized by day, or dropped); card ID / card aggregates allowed or not, and why._

## Q4. Transaction amounts

How skewed is `TransactionAmt`, and how concentrated are fraud dollars?

Decides: log transform for the MLP and logistic regression, and how sensitive dollar loss will be to a handful of large frauds.

In [ ]:
display(train.groupby(TARGET)[AMOUNT_COL].describe(percentiles=[0.5, 0.9, 0.99]).rename(index={0: "legit", 1: "fraud"}))

fig, ax = plt.subplots(figsize=(7, 3))
bins = np.linspace(0, np.log10(train[AMOUNT_COL].max()) + 0.1, 50)
for label, color, name in [(0, LEGIT, "legit"), (1, FRAUD, "fraud")]:
    ax.hist(np.log10(train.loc[train[TARGET] == label, AMOUNT_COL]), bins=bins, density=True,
            histtype="step", color=color, linewidth=2, label=name)
ax.set(title="Transaction amount by class (train)", xlabel="log10(TransactionAmt)", ylabel="density")
ax.legend(frameon=False)
plt.show()

In [ ]:
amt = train.loc[train[TARGET] == 1, AMOUNT_COL].sort_values(ascending=False)
for q in (0.01, 0.1):
    k = max(1, int(len(amt) * q))
    print(f"top {q:.0%} of frauds by amount hold {amt.iloc[:k].sum() / amt.sum():.1%} of fraud dollars")

**Decision (Q4):** _amount transform (e.g. log1p) for MLP / LR; note on dollar-loss sensitivity._

## Q5. Fraud rate over time

Is the shift from train (3.38%) to validation (3.90%) gradual or sudden?

Decides: calibration approach (fit on validation, and whether a recent window matters) and a first view of drift.

Weeks with under half the median volume (partial weeks at the edges of the data) are drawn hollow; their rates are noisy.

In [ ]:
week = (df[TIME_COL] // (7 * 86400)).rename("week")
weekly = df.groupby(week).agg(n=(TARGET, "size"), rate=(TARGET, "mean"))
boundary = val[TIME_COL].min() / (7 * 86400)
partial = weekly["n"] < 0.5 * weekly["n"].median()

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(weekly.index, weekly["rate"] * 100, color=LEGIT, marker="o", markersize=4)
axes[0].set(title="Weekly fraud rate", ylabel="fraud rate (%)")
axes[1].plot(weekly.index, weekly["n"] / 1000, color=LEGIT, marker="o", markersize=4)
axes[1].set(title="Weekly transaction volume", ylabel="transactions (k)", xlabel="week since start")
for ax, col, scale in [(axes[0], "rate", 100), (axes[1], "n", 1 / 1000)]:
    ax.plot(weekly.index[partial], weekly.loc[partial, col] * scale, "o", markersize=7,
            markerfacecolor="#fcfcfb", markeredgecolor=LEGIT, markeredgewidth=2)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.axvline(boundary, color=MUTED, linestyle="--", linewidth=1)
axes[0].annotate("train | val", (boundary, axes[0].get_ylim()[1]), ha="center", va="bottom", color=MUTED)
plt.tight_layout()
plt.show()

**Decision (Q5):** _calibration method and data (e.g. isotonic or Platt on validation); first note on drift._

## Q6. Input width for the MLP

DP-SGD adds noise to every gradient coordinate, so a much wider input costs more under privacy. Estimate the width under candidate preprocessing, using results from Q1 to Q3.

Decides: the feature set and encoding shared by the non-private and DP MLPs.

In [ ]:
numeric = [c for c in FEATURES if c not in CATEGORICAL and pd.api.types.is_numeric_dtype(df[c])]
for drop_thr in (0.5, 0.9, 1.0):
    kept = [c for c in numeric if miss[c] <= drop_thr]
    per_col = sum(miss[c] > 0 for c in kept)
    per_block = blocks.loc[blocks["miss_rate"] <= drop_thr, "n_cols"].size
    print(f"drop >{drop_thr:.0%} missing: numeric {len(kept):4d}"
          f" + indicators per column {per_col:4d} (or per block {per_block:3d}) + categorical width from Q2")

**Decision (Q6):** _final feature set, encoding, and approximate input width; confirm it is shared by both MLPs._

---

## Summary to copy into `docs/decisions.md`

| Question | Decision | Reason |
|---|---|---|
| Q1 Missingness | | |
| Q2 Categoricals | | |
| Q3 Leakage / time | | |
| Q4 Amounts | | |
| Q5 Fraud rate over time | | |
| Q6 MLP input width | | |